# Extract Belief Geometries — Qwen 3.5 9B

In [1]:
import os
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import gc
import numba
from tqdm import tqdm
from sklearn.model_selection import train_test_split

sns.set_context('notebook')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
np.random.seed(42); torch.manual_seed(42)

SEQ_LEN = 20_000
PROBE_START = 15_000
N_SEEDS = 10
TRAIN_FRAC = 0.2
# LAYERS set after model load
CHUNK_SIZE = 4096

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


## Model

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3.5-9B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    attn_implementation='sdpa', device_map='auto')
model.eval()

N_LAYERS = len(model.model.layers)
LAYERS = list(range(N_LAYERS))
print(f'Model: {MODEL_NAME}')
print(f'Layers: {N_LAYERS}, hidden_size: {model.config.hidden_size}')

TOKEN_NAMES_2 = np.array(['F', 'Q'])
TOKEN_NAMES_3 = np.array(['F', 'Q', 'V'])
TOK_IDS_2 = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in TOKEN_NAMES_2]
TOK_IDS_3 = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in TOKEN_NAMES_3]
print(f'2-token IDs: {dict(zip(TOKEN_NAMES_2, TOK_IDS_2))}')
print(f'3-token IDs: {dict(zip(TOKEN_NAMES_3, TOK_IDS_3))}')

# Verify single-token encoding
for name, tid in zip(TOKEN_NAMES_2, TOK_IDS_2):
    decoded = tokenizer.decode([tid])
    print(f'  {name} -> id={tid} -> decoded="{decoded}"')
for name, tid in zip(TOKEN_NAMES_3, TOK_IDS_3):
    decoded = tokenizer.decode([tid])
    print(f'  {name} -> id={tid} -> decoded="{decoded}"')


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model: Qwen/Qwen3.5-9B
Layers: 32, hidden_size: 4096
2-token IDs: {np.str_('F'): 426, np.str_('Q'): 1167}
3-token IDs: {np.str_('F'): 426, np.str_('Q'): 1167, np.str_('V'): 629}
  F -> id=426 -> decoded=" F"
  Q -> id=1167 -> decoded=" Q"
  F -> id=426 -> decoded=" F"
  Q -> id=1167 -> decoded=" Q"
  V -> id=629 -> decoded=" V"


## Infrastructure

In [3]:
def stationary_distribution(T_matrices):
    T_full = sum(T_matrices)
    eigvals, eigvecs = np.linalg.eig(T_full.T)
    idx = np.argmin(np.abs(eigvals - 1.0))
    pi = np.real(eigvecs[:, idx])
    return pi / pi.sum()

def sample_hmm_sequence(T_matrices, pi, seq_len, seed=None):
    rng = np.random.default_rng(seed)
    n_states, n_tokens = len(pi), len(T_matrices)
    state = rng.choice(n_states, p=pi)
    tokens = []
    for _ in range(seq_len):
        tp = np.array([T_matrices[z][state].sum() for z in range(n_tokens)])
        tp /= tp.sum()
        z = rng.choice(n_tokens, p=tp)
        tokens.append(z)
        nsp = T_matrices[z][state] / T_matrices[z][state].sum()
        state = rng.choice(n_states, p=nsp)
    return np.array(tokens)

def tokens_to_prompt(tokens, token_names, sep=' '):
    return sep + sep.join(token_names[t] for t in tokens)

def tokenize_prompt(prompt):
    return tokenizer.encode(prompt, return_tensors='pt', truncation=False)

def match_positions(input_ids, tok_ids):
    ids = input_ids[0].cpu().numpy()
    tok_id_set = {tid: zi for zi, tid in enumerate(tok_ids)}
    pos, tok = [], []
    for i, tid in enumerate(ids):
        if tid in tok_id_set:
            pos.append(i); tok.append(tok_id_set[tid])
    return np.array(pos), np.array(tok)

@numba.njit(cache=True)
def full_bayesian_beliefs_numba(tokens, T_stack, pi):
    n = len(tokens)
    n_states = len(pi)
    beliefs = np.zeros((n, n_states))
    b = pi.copy()
    for t in range(n):
        b = b @ T_stack[tokens[t]]
        s = 0.0
        for j in range(n_states):
            s += b[j]
        if s > 0:
            for j in range(n_states):
                b[j] /= s
        for j in range(n_states):
            beliefs[t, j] = b[j]
    return beliefs

def extract_and_probe_multi(input_ids, pos_indices, n_matched, targets_dict, seed):
    """Forward pass with hooks → R² for all layers, multiple targets.
    
    targets_dict: {'real': y_array, 'shuffle': y_array, 'random': y_array}
    Returns: {target_name: {layer: r2}}
    """
    seq_len = input_ids.shape[1]
    past_kv = None

    late_mask = np.arange(n_matched) >= PROBE_START
    late_model_pos = pos_indices[late_mask]
    n_late = int(late_mask.sum())

    acts = {l: [] for l in LAYERS}

    first_hooked_chunk = None
    for start in range(0, seq_len, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, seq_len)
        chunk = input_ids[:, start:end].to(device)
        need_hooks = (end > PROBE_START)
        hooks = []
        chunk_acts = {}

        if need_hooks:
            if first_hooked_chunk is None:
                first_hooked_chunk = start
            for l in LAYERS:
                def make_hook(li):
                    def fn(module, inp, out):
                        h = out[0] if isinstance(out, tuple) else out
                        chunk_acts[li] = h[0]
                    return fn
                hooks.append(model.model.layers[l].register_forward_hook(make_hook(l)))

        with torch.no_grad():
            out = model.model(chunk, past_key_values=past_kv, use_cache=True)

        for h in hooks:
            h.remove()

        if need_hooks:
            chunk_pos = np.arange(start, end)
            needed = np.isin(chunk_pos, late_model_pos)
            if needed.any():
                idx = torch.tensor(np.where(needed)[0], device=device)
                for l in LAYERS:
                    acts[l].append(chunk_acts[l][idx])

        past_kv = out.past_key_values
        del out, chunk_acts; torch.cuda.empty_cache()

    del past_kv; torch.cuda.empty_cache()

    # Train/test split (same for all targets)
    idx_tr, idx_te = train_test_split(
        np.arange(n_late), train_size=TRAIN_FRAC, random_state=seed)

    # Precompute Y tensors for all targets
    Y_targets = {}
    for tname, y in targets_dict.items():
        Y_targets[tname] = (
            torch.tensor(y[idx_tr], device=device, dtype=torch.float32),
            torch.tensor(y[idx_te], device=device, dtype=torch.float32),
        )

    # Probe each layer
    results = {tname: {} for tname in targets_dict}
    for l in LAYERS:
        X = torch.cat(acts[l], dim=0).float()
        X_tr = X[idx_tr]
        X_te = X[idx_te]
        P = torch.linalg.pinv(X_tr)  # shared across targets

        for tname in targets_dict:
            Y_tr, Y_te = Y_targets[tname]
            W = P @ Y_tr
            pred = X_te @ W
            ss_res = ((Y_te - pred) ** 2).sum().item()
            ss_tot = ((Y_te - Y_te.mean(0)) ** 2).sum().item()
            results[tname][l] = 1.0 - ss_res / ss_tot

        del X, X_tr, X_te, P

    del acts, Y_targets; torch.cuda.empty_cache()
    return results

# Warmup numba
_d = np.random.rand(3, 4, 4); _p = np.array([0.25, 0.25, 0.25, 0.25])
_ = full_bayesian_beliefs_numba(np.array([0, 1, 2], dtype=np.int64), _d, _p)
del _d, _p
print('Infrastructure loaded. Numba compiled.')


Infrastructure loaded. Numba compiled.


## HMM Definitions

In [4]:
# ===== Mess3 (3 tokens: A, B, C) =====

def mess3_matrices(a, x):
    b = (1 - a) / 2
    y = 1 - 2 * x
    ay, bx, by, ax = a*y, b*x, b*y, a*x
    return [
        np.array([[ay, bx, bx], [ax, by, bx], [ax, bx, by]]),
        np.array([[by, ax, bx], [bx, ay, bx], [bx, ax, by]]),
        np.array([[by, bx, ax], [bx, by, ax], [bx, bx, ay]]),
    ]

# ===== Wing (2 tokens: F, Q) =====

def wing_matrices(x, y):
    b = (1 - x) / 2
    return [
        np.array([[0, b, 0], [0, y*x, 0.5*b], [b, 0, 0]]),
        np.array([[x, 0, b], [b, (1-y)*x, 0.5*b], [0, b, x]]),
    ]

# ===== Strata (2 tokens: F, Q) =====

def strata_matrices(a, t0, t1):
    b = (1 - a) / 2
    return [
        np.array([[t0*a, 0, 0], [0, t1*a, 0], [0, 0, 0]]),
        np.array([[(1-t0)*a, b, b], [b, (1-t1)*a, b], [b, b, a]]),
    ]

# ===== Arch (3 tokens: A, B, C) =====

def arch_matrices(a):
    b = (1 - a) / 3
    return [
        np.array([
            [0.8*a, 0, 0, 0],
            [0, 0.2*a, 0, 0],
            [0, 0, 0.4*a, 0],
            [0, 0, 0, 0.6*a],
        ]),
        np.array([
            [0, 0, 0, 0],
            [0, 0.4*a, 0, 0.4*b],
            [0, 0, 0.3*a, 0],
            [0, 0, 0, 0.16*a],
        ]),
        np.array([
            [0.2*a, b, b, b],
            [b, 0.4*a, b, 0.6*b],
            [b, b, 0.3*a, b],
            [b, b, b, 0.24*a],
        ]),
    ]

# ===== Spiral (2 tokens: F, Q) =====

def spiral_matrices(a):
    return [
        np.array([
            [0.2*a,       0,   0     ],
            [0,           0,   0     ],
            [0.25*(1-a),  0,   0.5*a ],
        ]),
        np.array([
            [0.8*a,       1-a, 0     ],
            [0,           a,   1-a   ],
            [0.75*(1-a),  0,   0.5*a ],
        ]),
    ]

print('All HMM functions defined.')


All HMM functions defined.


## HMM Registry

In [5]:
HMMS = {
    'Wing': {
        'fn': wing_matrices,
        'order_one_fn': lambda *p: wing_order_one(*p),
        'order_zero_fn': lambda *p: wing_order_zero(*p),
        'params': [(0.98, 0.4)],
        'label_fn': lambda p: f'x={p[0]}, y={p[1]}',
        'token_names': np.array(['F', 'Q']),
    },
    'Strata': {
        'fn': strata_matrices,
        'order_one_fn': lambda *p: strata_order_one(*p),
        'order_zero_fn': lambda *p: strata_order_zero(*p),
        'params': [(0.97, 0.38, 0.54)],
        'label_fn': lambda p: f'a={p[0]}, t0={p[1]}, t1={p[2]}',
        'token_names': np.array(['F', 'Q']),
    },
    'Arch': {
        'fn': arch_matrices,
        'order_one_fn': lambda *p: arch_order_one(*p),
        'order_zero_fn': lambda *p: arch_order_zero(*p),
        'params': [(0.9,)],
        'label_fn': lambda p: f'a={p[0]}',
        'token_names': np.array(['F', 'Q', 'V']),
    },
    'Mess3': {
        'fn': mess3_matrices,
        'order_one_fn': lambda a, x: mess3_order_one(a, x),
        'order_zero_fn': None,
        'params': [(0.01, 0.02)],
        'label_fn': lambda p: f'a={p[0]}, x={p[1]}',
        'token_names': np.array(['F', 'Q', 'V']),
    },
}
for name, cfg in HMMS.items():
    T = cfg['fn'](*cfg['params'][0])
    print(f'{name:>8}: {len(cfg["params"])} params, {len(T)} tokens, {T[0].shape[0]} states')


    Wing: 1 params, 2 tokens, 3 states
  Strata: 1 params, 2 tokens, 3 states
    Arch: 1 params, 3 tokens, 4 states
   Mess3: 1 params, 3 tokens, 3 states


## Load R² CSV to find best layers

In [7]:
r2_df = pd.read_csv('r2_qwen35_9b.csv')
print(f'Loaded {len(r2_df)} rows')
print(f'HMMs: {sorted(r2_df["hmm"].unique())}')


Loaded 48000 rows
HMMs: ['Arch', 'Mess3', 'Spiral', 'Strata', 'Wing']


## Extract activations at best layer + compute predicted beliefs

In [14]:
geom_results = {}
for hmm_name, cfg in HMMS.items():
    param = cfg['params'][0]
    label = cfg['label_fn'](param)
    token_names = cfg['token_names']
    tok_ids = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in token_names]
    T_real = cfg['fn'](*param)
    T_stack = np.stack(T_real)
    pi_real = stationary_distribution(T_real)
    r2_sub = r2_df[(r2_df['hmm'] == hmm_name) & (r2_df['param'] == label) & (r2_df['target'] == 'real')]
    best_layer = int(r2_sub.groupby('layer')['R2'].mean().idxmax())
    print(f'{hmm_name} ({label}): best layer = {best_layer}')
    tokens = sample_hmm_sequence(T_real, pi_real, SEQ_LEN, seed=0)
    beliefs = full_bayesian_beliefs_numba(tokens.astype(np.int64), T_stack, pi_real)
    prompt = tokens_to_prompt(tokens, token_names)
    input_ids = tokenize_prompt(prompt)
    pos_indices, _ = match_positions(input_ids, tok_ids)
    n_matched = min(len(tokens), len(pos_indices))
    late_mask = np.arange(n_matched) >= PROBE_START
    late_model_pos = pos_indices[:n_matched][late_mask]
    n_late = int(late_mask.sum())
    y_true = beliefs[PROBE_START:PROBE_START + n_late]
    acts = []
    past_kv = None
    print(f'  Forward pass...', end='', flush=True)
    for start in range(0, input_ids.shape[1], CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, input_ids.shape[1])
        chunk = input_ids[:, start:end].to(device)
        need_acts = (end > PROBE_START)
        chunk_act = {}
        hooks = []
        if need_acts:
            def make_hook(li):
                def fn(module, inp, out):
                    h = out[0] if isinstance(out, tuple) else out
                    chunk_act[li] = h[0]
                return fn
            hooks.append(model.model.layers[best_layer].register_forward_hook(make_hook(best_layer)))
        with torch.no_grad():
            out = model.model(chunk, past_key_values=past_kv, use_cache=True)
        for h in hooks:
            h.remove()
        if need_acts and best_layer in chunk_act:
            chunk_positions = np.arange(start, end)
            needed = np.isin(chunk_positions, late_model_pos)
            if needed.any():
                idx = np.where(needed)[0]
                acts.append(chunk_act[best_layer][idx].float().cpu())
        past_kv = out.past_key_values
        del out, chunk_act; torch.cuda.empty_cache()
        print('.', end='', flush=True)
    del past_kv; torch.cuda.empty_cache()
    acts_np = torch.cat(acts, dim=0).numpy()
    del acts
    print(f' done ({acts_np.shape[0]} positions)')
    X = np.hstack([acts_np, np.ones((len(acts_np), 1))])
    W = np.linalg.pinv(X) @ y_true
    y_pred = X @ W
    r2 = 1 - ((y_true - y_pred)**2).sum() / ((y_true - y_true.mean(0))**2).sum()
    print(f'  Probe R² = {r2:.4f}')
    geom_results[hmm_name] = {
        'beliefs_true': y_true,
        'beliefs_pred': y_pred,
        'n_states': y_true.shape[1],
        'best_layer': best_layer,
        'param': label,
    }
    del acts_np; gc.collect(); torch.cuda.empty_cache()
print('\nDone.')

Wing (x=0.98, y=0.4): best layer = 22
  Forward pass........ done (5000 positions)
  Probe R² = 0.9999
Strata (a=0.97, t0=0.38, t1=0.54): best layer = 30
  Forward pass........ done (5000 positions)
  Probe R² = 0.9997
Arch (a=0.9): best layer = 2
  Forward pass........ done (5000 positions)
  Probe R² = 0.9985
Mess3 (a=0.01, x=0.02): best layer = 5
  Forward pass........ done (5000 positions)
  Probe R² = 0.9998

Done.


## Save

In [15]:
save_dict = {}
for hmm_name, gd in geom_results.items():
    save_dict[f'{hmm_name}_true'] = gd['beliefs_true']
    save_dict[f'{hmm_name}_pred'] = gd['beliefs_pred']
    save_dict[f'{hmm_name}_nstates'] = np.array(gd['n_states'])
    save_dict[f'{hmm_name}_bestlayer'] = np.array(gd['best_layer'])
    save_dict[f'{hmm_name}_param'] = np.array(gd['param'])

np.savez(os.path.join(RESULTS_DIR, 'geom_qwen35_9b.npz'), **save_dict)
print('Saved geom_qwen35_9b.npz')
for k, v in save_dict.items():
    if isinstance(v, np.ndarray) and v.ndim > 0:
        print(f'  {k}: {v.shape}')
    else:
        print(f'  {k}: {v}')


Saved geom_qwen35_9b.npz
  Wing_true: (5000, 3)
  Wing_pred: (5000, 3)
  Wing_nstates: 3
  Wing_bestlayer: 22
  Wing_param: x=0.98, y=0.4
  Strata_true: (5000, 3)
  Strata_pred: (5000, 3)
  Strata_nstates: 3
  Strata_bestlayer: 30
  Strata_param: a=0.97, t0=0.38, t1=0.54
  Arch_true: (5000, 4)
  Arch_pred: (5000, 4)
  Arch_nstates: 4
  Arch_bestlayer: 2
  Arch_param: a=0.9
  Mess3_true: (5000, 3)
  Mess3_pred: (5000, 3)
  Mess3_nstates: 3
  Mess3_bestlayer: 5
  Mess3_param: a=0.01, x=0.02
